<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/03_constrained_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Constrained optimization

In [ ]:
from typing import Callable
import jax.numpy as jnp
import matplotlib.pyplot as plt
import functools
import jax
import numpy as np
import cvxpy as cp


Constrained optimization involves finding the minimum or maximum of an objective function subject to one or more constraints on the variables. These constraints can be equalities or inequalities that restrict the set of feasible solutions. Such problems are common in engineering, economics, and machine learning, where solutions must satisfy specific requirements or limitations.


- **Resource allocation:** In manufacturing, the total amount of raw materials used cannot exceed available supply (inequality constraint).
- **Portfolio optimization:** The sum of investment weights must equal 1 (equality constraint), and each weight must be non-negative (inequality constraint).
- **Engineering design:** The stress on a bridge component must not exceed a safety threshold (inequality constraint).
- **Logistics:** Delivery routes must start and end at a depot (equality constraint), and vehicle capacities cannot be exceeded (inequality constraint).

With constrained optimization, we cannot just throw vanilla gradient descent at the problem because the gradient may push the solution towards infeasible regions, and it does not know when to stop descending.



## Log-barrier method for solving constrained optimization problems

*(Credit: Oliver Sheridan, Spring 2025)*

In this problem, we turn a constrained optimization problem into an unconstrained one by using the log-barrier method.

Consider the objective function $$f(x) = (x + 2)^2 + 5\tanh (x)$$

The value of $x$ which minimizes $f(x)$ is $x^* = -2.13578$, and $f(x^*) = -4.84389$.

Fun fact: with $f(x) = (x + 2)^2 + 5\tanh (x)$, then $f'(x)$ is a transcendental function, and $f'(x) = 0$ _can't_ be solved algebraically. If your gradient descent code worked, congratulations! You've written code that numerically solved a math problem in milliseconds that can't be solved analytically at all.


In [ ]:
def objective_function(x: jnp.ndarray) -> jnp.ndarray:
    """Objective function to minimize."""
    return (x + 2) ** 2 + 5 * jnp.tanh(x)

xs = jnp.linspace(-6, 5, 100)
plt.plot(xs, objective_function(xs), label='Objective Function')
plt.grid(alpha=0.3)
plt.legend()

### (a) Implement gradient descent on the function to find the minimum.

Write a function performing gradient descent, and plot the results

In [ ]:
# TODO in class
def gradient_descent(
    func: Callable[[jnp.ndarray, jnp.ndarray], jnp.ndarray],
    guess: jnp.ndarray,
    learning_rate: float = 0.001,
    num_steps: int = 10000
):
pass

In [ ]:
guess = 5.
learning_rate = 0.01
num_steps = 100
guesses, values, sol = gradient_descent(objective_function, guess, learning_rate, num_steps)

In [ ]:
xs = jnp.linspace(-6, 5, 100)
plt.plot(xs, objective_function(xs), label='Objective Function')
plt.scatter(guesses, values, color='red', label='Gradient Descent Steps')
plt.scatter(sol, objective_function(sol), color='green', label='Estimated Minimum', zorder=5)
plt.grid(alpha=0.3)
plt.legend()

### (b) Applying log-barrier for solving constrained optimization problems

Now suppose we want to add a _constraint_ to our optimization problem; that is, a restriction on the set of $x$ values we'll consider acceptable. In particular, in this case we'll say that we only want values of $x$ such that, for some specified _constraint function_ $g(x)$, we have $g(x) < 0$; this is called an _inequality constraint_.

The gradient descent algorithm described above has no way to enforce a constraint like this; $x$ is allowed to wander wherever the gradient takes it. Thus we must modify the algorithm to allow it to find constrained optima under inequality constraints. There are several ways to do this, but in this problem we will be using the _log-barrier method_.

In this method, we construct from the objective function $f(x)$ a different objective function $\phi (x)$ that has the following properties:
 - When $x$ is such that $g(x)$ is far away from $0$, then $\phi(x) \approx f(x)$, so that minimizing $\phi(x)$ approximately minimizes $f(x)$.
 - $\phi(x)$ must grow to infinity as $g(x)$ approaces $0$. This will prevent $x$ from crossing over the boundary of $g(x) < 0$.

To accomplish these goals, we construct $\phi(x)$ like so:
$$
\phi(x) = f(x) - \frac{1}{t}\ln(-g(x)),
$$
where $t$ is a weighting parameter that we choose.

Suppose we want to minimize the same objective function as before, $f(x) = (x + 2)^2 + 5\tanh (x)$, but now we want a constraint $x > 1$.


Implement the new objective function with the log barrier applied on the constraint.

Hint: for your `phi` function to work well with the next parts of this problem, be sure to use `jnp.log` instead of `np.log`.

In [ ]:
# TODO with peer
def objective_log_barrier(x: jnp.ndarray, t: float = 1.0) -> jnp.ndarray:
    """Objective function with log barrier for the constraint x <= 2."""
    pass

### (c) Plot $\phi(x)$.

Plot $\phi(x)$ for $t = 0.5,\ 2,\ 5$. Comment on how $\phi(x)$ changes with changing $t$.

Note that since $\ln(y)$ is not defined (in the real numbers) for $y \le 0$, the domain of $\phi(x)$ is restricted to $x > 1$.


In [ ]:
# TODO with peer

### (c) Run gradient descent and visualize results

Run gradient descent for the various $t$ values, and visualize the results

In [ ]:
# TODO with peer

### (d) Multiple constraints and higher dimensions?


In [ ]:
# TODO with peers


How would you use the log-barrier method for problems with higher dimensions and multiple constraints? How would this affect how easy or hard the problem is to solve? Why?

- **Formulation (multiple constraints, higher-D)**:
  - For inequalities $ g_i(x) < 0 $, use a summed barrier: $ \phi_t(x) = f(x) − (1/t)\sum_i log(−g_i(x)) $. Domain is the interior $ \{x | g_i(x) < 0 \forall i\} $.
  - For equalities $ h_j(x) = 0 $, handle via KKT (Lagrange multipliers) or eliminate with a feasible parameterization (null-space method).

- **Scaling to higher dimensions (n) and many constraints (m)**:
  - Each Newton step solves a linear system with Hessian of $ \phi_t $ dense; better with sparsity/structure. Barrier adds $ \sum (\nabla g_i \nabla g_i^T)/(−g_i)^2 $ and $ \nabla^2g_i $ terms, increasing cost with m.
  - More constraints tighten the feasible interior, making $ \phi_t $ steeper near boundaries and conditioning worse; step sizes shrink and more iterations may be needed.

- **Difficulty impact**:
  - Convex problems: interior-point with log barriers remains polynomial-time and very effective; larger n, m primarily raise per-iteration cost and conditioning challenges.
  - Nonconvex problems: barriers do not remove nonconvexity—only local guarantees; risk of getting stuck or following a non-global central path.

### (e) Other approaches?

In [ ]:
# TODO with peers/self

What are things you could do to cast a constrained optimization problem into an unconstrained on?

- **Barrier methods**: Add interior (log) barriers for inequalities:
\begin{aligned}
\phi_t(x) = f(x) − (1/t)\sum log(−g_i(x)).
\end{aligned}
- **Penalty methods**: Add penalties to the objective, e.g., quadratic for equalities and hinge/quadratic for inequalities: 

\begin{aligned}
f(x) + \rho \sum [g_i(x)]_+^2 + \rho \sum h_j(x)^2.
\end{aligned}

- **Augmented Lagrangian**: Combine Lagrange multipliers with quadratic penalties to reduce ill-conditioning vs pure penalties.

- **Exact penalties**: Use L1 penalty on constraints (with sufficiently large $ \rho $), yielding exact satisfaction at the solution.

- **Parameterization**: Reparameterize variables to embed constraints (e.g., $ x = \exp(y) $ for positivity; affine null-space for equalities).

- **Projection-free transforms**: Use soft constraints via smooth approximations (e.g., softplus) when differentiability is needed.

- **Box constraints to unconstrained**: Map bounded variables with logistic/tanh transforms to ℝ.

- **Feasible-start transforms**: Start from strictly feasible x and maintain feasibility via barrier or change of variables.

## `cvxpy`

In this problem we will explore the basics of `cvxpy`, a Python package for solving convex optimization problems. `cvxpy` has a good tutorial [here](https://www.cvxpy.org/tutorial/intro/index.html), so read that page before proceeding with this problem (the section on "parameters", while useful, is not important for this problem, so consider that section optional for now).

### (a)
Consider a vector variable $x = [x_1,  x_2,  x_3]^T$. Use `cvxpy` to compute the minimizer of the following objective function
$$
x_1^2 + 2x_2^2 + 3.5x_3^2,
$$

subject to the constraint

$$
\begin{bmatrix}
0.707 & 0.707 & 0 \\
-1 & 0 & 0 \\
0 & -1 & 0 \\
0 & 0 & -1
\end{bmatrix}x \le \begin{bmatrix}
2 \\
-1 \\
-1 \\
-3
\end{bmatrix}.
$$

Print the optimal $x$ and the optimal value of the objective function.

Notice how much easier it is to use `cvxpy` than to write our own optimization algorithm from scratch!

In [ ]:
# TODO in class


In [ ]:
# TODO with peers/self



### (b)

Suppose instead that we want to select a sequence of control inputs, say $u_0, \ldots, u_T$, subject to linear dynamics $x_{t+1} = Ax_t + Bu_t$ and control limits $|u_t| < u_{\max}$, and minimizing the total control effort $\sum_{t=0}^T u_t^Tu_t$. Assume the current state is $x_0$. Describe at a high-level how you may go about formulating the optimization problem using `cvxpy`.

For instance, describe how you would define `Variable`, `constraints`, `Parameters`, and `objective`, and any characteristics about them.

- **Variables**:
  - Decision variables for inputs: `U = cp.Variable((m, T+1))` (or vectorized `cp.Variable(m*(T+1))`). Optionally include states `X = cp.Variable((n, T+1))` if not eliminated.

- **Parameters/Data**:
  - System matrices `A, B`, horizon `T`, input limit `u_max`, initial state `x0` as `cp.Parameter`/constants.

- **Dynamics constraints**:
  - If keeping states: `X[:, t+1] == A@X[:, t] + B@U[:, t]` for t = 0..T−1, and `X[:,0] == x0`.
  - Alternatively eliminate states: unroll dynamics to express each `X[:, t]` as affine in `x0` and past `U`, then keep only input variables.

- **Input constraints**:
  - Box limits: `cp.norm_inf(U[:, t]) <= u_max` or elementwise `-u_max <= U <= u_max`.

- **Objective**:
  - Quadratic effort: minimize `sum_{t=0}^T U[:, t].T @ U[:, t]` i.e., `cp.sum_squares(U)` (optionally add state penalties `cp.sum_squares(X)` for LQR-like costs).

- **Problem**:
  - `prob = cp.Problem(cp.Minimize(objective), constraints)`; solve with a convex QP solver. This is convex (quadratic objective, linear constraints).